# DIII-D 162940 — profile scaling, then cheaseBS

Two ways of scaling the same discharge, plotted side by side before anything is
solved, following `TPED/projects/discharge_tools/examples/discharge_scaling.ipynb`.

| Method | Model | What the knob does |
|--------|-------|--------------------|
| `apply_omt` / `apply_omne` | power-law gradient | multiplies the log-gradient in the pedestal window; pins the profile at `rhot_midped` |
| `apply_mtanh_full` | Stefanikova 2016 | rescales a fitted parameter (`scale_height`, `scale_width`, ...) and rebuilds the profile from the fit |

Look at the plots and the summary tables first. `RUN_CHEASEBS` at the bottom is
off by default; flip it once the scalings look like what you meant.


## Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from TPED.projects.discharge_tools.src.discharge_data import DischargeData
from TPED.projects.discharge_tools.src.discharge_physics import DischargePhysics
from TPED.projects.discharge_tools.src.transforms.mtanh_transforms import fit_mtanh_full

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

### Case and knobs — edit these

`RHOT_TOPPED` / `RHOT_MIDPED` are the 162940 pedestal measured from an
`mtanh_full` fit to `pe`: `midped = b_pos`, `topped = b_pos - 2*b_width`. That
pedestal is narrow (full width 0.033), so a gradient alpha near 1 moves very
little — which is why the first campaign's omn/omt points barely changed the
equilibrium. The alpha list below spans the example notebook's range instead.

In [ ]:
CASE_DIR = os.path.expandvars("$SCRATCH/DIIID162940/DIIID162940")

# 162940 pedestal, from the mtanh_full fit to pe.
RHOT_TOPPED, RHOT_MIDPED = 0.954, 0.971

# Gradient multipliers for apply_omt / apply_omne.
ALPHAS = [0.5, 0.75, 1.0, 1.5, 2.0]

# Fit-parameter multipliers for apply_mtanh_full.
MTANH_SCALES = [0.8, 0.9, 1.0, 1.1, 1.2]

RUN_CHEASEBS = False          # <-- flip to solve equilibria for the cases below
CHEASEBS_OUTROOT = os.path.expandvars("$SCRATCH/cheasebs_scaling_notebook")

In [ ]:
phys = DischargePhysics(DischargeData(input_dir=CASE_DIR))
print("variables:", list(phys.ds.data_vars))
print("gfile     :", phys._tree["raw/gfile"].dataset.attrs.get("filename"))

### Plotting helper

From the example notebook — overlays base vs scaled profiles.

In [ ]:
def get_vals(phys, var):
    """Extract numpy array, stripping pint units if present."""
    da = phys.ds[var]
    return da.pint.magnitude if hasattr(da, "pint") else da.values


def compare_profiles(phys_list, labels, vars=("Te", "ne"), title="",
                     rho_range=(0.0, 1.0)):
    """Overlay profiles from multiple DischargePhysics objects."""
    fig, axes = plt.subplots(1, len(vars), figsize=(4.5 * len(vars), 4))
    if len(vars) == 1:
        axes = [axes]
    colors = plt.cm.tab10(np.linspace(0, 0.8, len(phys_list)))
    linestyles = ["-"] + ["--", "-.", ":"] * len(phys_list)

    for p, label, color, ls in zip(phys_list, labels, colors, linestyles):
        rhot = p.rhot.values
        mask = (rhot >= rho_range[0]) & (rhot <= rho_range[1])
        for ax, var in zip(axes, vars):
            ax.plot(rhot[mask], get_vals(p, var)[mask], label=label,
                    color=color, ls=ls, lw=1.8)
            ax.set_xlabel(r"$\rho_{\mathrm{tor}}$")
            ax.set_title(var)
            ax.set_xlim(*rho_range)
            ax.grid(alpha=0.25)
    axes[0].legend(fontsize=8)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    return fig

### How big is the change, really

`L = -y/(dy/drho)` at the pedestal, and the thermal pressure integrated over
rho. A scaling that leaves `dp_int` at zero leaves the equilibrium alone, and
cheaseBS will return the profile it started from.

In [ ]:
EV = 1.602176634e-19


def pressure(p):
    out = get_vals(p, "ne") * get_vals(p, "Te")
    for n, t in (("ni", "Ti"), ("nz", "Tz")):
        if n in p.ds and t in p.ds:
            out = out + get_vals(p, n) * get_vals(p, t)
    return out * EV


def summary(base, cases, labels, radius=0.99):
    """One row per case: L at `radius`, and pressure change vs base."""
    x = base.rhot.values
    o = np.argsort(x)
    p0, L0 = pressure(base), np.asarray(base.gradient_length("Te").values, float)
    p0_int = float(np.trapezoid(p0[o], x[o]))
    L0_r = float(np.interp(radius, x[o], L0[o]))

    print(f"{'case':<22}{'L_Te':>9}{'L/L_base':>10}{'dp@0.5':>9}{'dp_int':>9}")
    for p, label in zip(cases, labels):
        L = np.asarray(p.gradient_length("Te").values, float)
        L_r = float(np.interp(radius, x[o], L[o]))
        pr = pressure(p)
        d_mid = np.interp(0.5, x[o], pr[o]) / np.interp(0.5, x[o], p0[o]) - 1
        d_int = float(np.trapezoid(pr[o], x[o])) / p0_int - 1
        print(f"{label:<22}{L_r:>9.5f}{L_r / L0_r:>10.2f}"
              f"{d_mid * 100:>8.1f}%{d_int * 100:>8.1f}%")

---
## 1. `apply_omt` / `apply_omne` — gradient scaling

Power-law on the log-gradient inside `[rhot_topped, rhot_midped]`, pinned at
`rhot_midped`. `alpha < 1` flattens, `alpha > 1` steepens. Density scaling
rewrites `ni` and `nz` through quasineutrality.

In [ ]:
omt_cases = [phys.apply_omt(alpha=a, rhot_midped=RHOT_MIDPED,
                           rhot_topped=RHOT_TOPPED) for a in ALPHAS]

compare_profiles(omt_cases, [f"alpha={a}" for a in ALPHAS], vars=("Te", "Ti"),
                 title="apply_omt — temperature gradient scaling",
                 rho_range=(0.90, 1.0))
plt.show()

In [ ]:
omne_cases = [phys.apply_omne(alpha=a, rhot_midped=RHOT_MIDPED,
                             rhot_topped=RHOT_TOPPED) for a in ALPHAS]

compare_profiles(omne_cases, [f"alpha={a}" for a in ALPHAS], vars=("ne", "ni"),
                 title="apply_omne — density gradient scaling",
                 rho_range=(0.90, 1.0))
plt.show()

print("quasineutrality error, base:", f"{phys.check_quasineutrality():.2e}")
for a, p in zip(ALPHAS, omne_cases):
    print(f"  alpha={a}: {p.check_quasineutrality():.2e}")

In [ ]:
# Both knobs together, which is what the cheaseBS cases below use.
omn_omt_cases = [phys.apply_omt(alpha=a, rhot_midped=RHOT_MIDPED,
                                rhot_topped=RHOT_TOPPED)
                     .apply_omne(alpha=a, rhot_midped=RHOT_MIDPED,
                                 rhot_topped=RHOT_TOPPED)
                 for a in ALPHAS]

summary(phys, omn_omt_cases, [f"omn_omt alpha={a}" for a in ALPHAS])

---
## 2. `apply_mtanh_full` — Stefanikova fit scaling

Fit once, reuse the fit at every scale. `scale_height` moves the pedestal top
`b_height`; `scale_width` moves the quarter-width `b_width`. Note the fit
reconstruction replaces the profile even at scale 1.0, so the unity case is not
the base profile.

In [ ]:
fits = {}
for var in ("Te", "Ti", "Tz", "ne"):
    if var in phys.ds:
        fits[var], rec = fit_mtanh_full(phys.ds, var, pedestal_weight=8.0)
        print(f"{var}: rms {rec['rms_relative']:.2%}  "
              f"b_pos={fits[var].b_pos:.4f}  b_width={fits[var].b_width:.5f}")

In [ ]:
rhot = phys.rhot.values
rhot_fine = np.linspace(0.7, 1.0, 500)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, var in zip(axes, ("Te", "ne")):
    mask = rhot >= 0.7
    ax.scatter(rhot[mask], get_vals(phys, var)[mask], s=10, color="steelblue",
               label="data", zorder=3)
    ax.plot(rhot_fine, fits[var](rhot_fine), color="tomato", lw=2, label="mtanh_full fit")
    ax.axvline(fits[var].b_pos, color="gray", ls=":", lw=1,
               label=f"b_pos={fits[var].b_pos:.3f}")
    ax.set_xlabel(r"$\rho_{\mathrm{tor}}$")
    ax.set_title(var)
    ax.set_xlim(0.7, 1.0)
    ax.legend(fontsize=8)
fig.suptitle("Stefanikova F_full fit", fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
def scale_mtanh(p, s, knob):
    """Apply one fit-parameter scale to every species, ne last for quasineutrality."""
    for var in ("Te", "Ti", "Tz"):
        if var in p.ds and var in fits:
            p = p.apply_mtanh_full(var, fit=fits[var], **{knob: s})
    return p.apply_mtanh_full("ne", fit=fits["ne"], enforce_quasineutrality=True,
                              qz=6.0, **{knob: s})


height_cases = [scale_mtanh(phys, s, "scale_height") for s in MTANH_SCALES]
width_cases = [scale_mtanh(phys, s, "scale_width") for s in MTANH_SCALES]

compare_profiles(height_cases, [f"x{s}" for s in MTANH_SCALES], vars=("Te", "ne"),
                 title="apply_mtanh_full — scale_height", rho_range=(0.7, 1.0))
plt.show()

compare_profiles(width_cases, [f"x{s}" for s in MTANH_SCALES], vars=("Te", "ne"),
                 title="apply_mtanh_full — scale_width", rho_range=(0.7, 1.0))
plt.show()

In [ ]:
summary(phys, height_cases, [f"height x{s}" for s in MTANH_SCALES])
print()
summary(phys, width_cases, [f"width x{s}" for s in MTANH_SCALES])

---
## 3. cheaseBS

Off unless `RUN_CHEASEBS` is True. Each case runs on scratch and is scored; the
reference profiles are the untransformed base, so the baseline decomposition
stays a fixed frame rather than moving with the scan.

CHEASE's own Grad-Shafranov iteration is what fails when a case is too far from
the source equilibrium (`NCON=-2`, "ITERATION OVER CURRENT PROFILE NOT
CONVERGED" in its log). The `dp_int` column above is the number to watch.

In [ ]:
CASES = {}
for a in ALPHAS:
    CASES[f"omn_omt_a{a}"] = phys.apply_omt(
        alpha=a, rhot_midped=RHOT_MIDPED, rhot_topped=RHOT_TOPPED
    ).apply_omne(alpha=a, rhot_midped=RHOT_MIDPED, rhot_topped=RHOT_TOPPED)
for s in MTANH_SCALES:
    CASES[f"mtanh_height_{s}"] = scale_mtanh(phys, s, "scale_height")

print(f"{len(CASES)} case(s):", ", ".join(CASES))

In [ ]:
if RUN_CHEASEBS:
    from TPED.config.config_helper import Config
    from TPED.projects.discharge_tools.src.cheasebs_runner import (
        CheasebsAcceptance, run_cheasebs_workflow)

    cfg = Config()
    cheasebs_dir = cfg.get_path("CHEASEBS_PATH")
    paths = dict(
        chease_binary=os.path.join(cfg.get_path("CHEASE_PATH"), "src-f90", "chease"),
        cheasebs_script=os.path.join(cheasebs_dir, "run_chease_iterative_profiles.py"),
        chease_namelist=os.path.join(cheasebs_dir, "chease_namelist"),
    )
    gfile = phys._tree["raw/gfile"].dataset.attrs
    gfile = os.path.join(gfile["directory"], gfile["filename"])
    baseline = os.path.join(CHEASEBS_OUTROOT, "baseline")

    results = {}
    for name, case in CASES.items():
        savedir = os.path.join(CHEASEBS_OUTROOT, name)
        print(f"\n=== {name} ===", flush=True)
        try:
            eqdsk, res = run_cheasebs_workflow(
                gfile_path=gfile, ds=case.ds, reference_ds=phys.ds,
                savedir=savedir, config_template="diiid_cheasebs_config.json",
                baseline_dir=baseline, acceptance=CheasebsAcceptance.production(),
                strict=False, return_acceptance=True, **paths)
            results[name] = res.to_dict()
        except Exception as exc:
            print(f"  FAILED: {type(exc).__name__}: {exc}")
            results[name] = {"error": f"{type(exc).__name__}: {exc}"}

    print(f"\n{'case':<22}{'accepted':>10}{'Ip_err':>10}{'iters':>7}")
    for name, r in results.items():
        if "error" in r:
            print(f"{name:<22}{'FAILED':>10}")
            continue
        ip = r.get("ip_error_rel")
        print(f"{name:<22}{str(r.get('accepted')):>10}"
              f"{(f'{ip:.2%}' if ip is not None else '--'):>10}"
              f"{str(r.get('cheasebs_iterations')):>7}")
else:
    print("RUN_CHEASEBS is False — set it True to solve the cases above.")